# exp05d - Pengembangan AR-LRX: gerbang terpelajar vs gerbang tabel

## Dari mana ide ini datang

Tabel segmen exp05b memberi petunjuk yang jelas. Pada V3, AR-LRX **kalah** dari
XGBoost polos di Selasa (+12,5%), Kamis (+13,7%), hari promo (+1,8%) dan toko kecil
(+5,6%), tetapi **menang besar** di Sabtu (-17,9%), Senin (-10,6%) dan Minggu (-7,2%).
Satu nilai `w` global jelas kompromi buruk untuk pola sebeda itu.

Ablasi bersih exp05b juga menunjukkan gerbang skalar praktis tidak menyumbang apa pun
(rata-rata +0,075% untuk tahap pertama yang baik). Jadi persoalannya bukan "gerbangnya
kurang disetel", melainkan **bentuk gerbangnya terlalu kaku**.

Eksperimen ini menguji dua cara memperbaikinya, dan sengaja mengadu keduanya.

| | Gagasan | Bentuk gerbang |
|---|---|---|
| **Seg** | `w` dipilih per segmen (promo / hari / hari x promo / kuartil toko) | tabel pencarian, konstan-sepotong |
| **Aug** | prediksi tahap pertama dijadikan **fitur** bagi tahap kedua | terpelajar, kontinu, bergantung fitur |

Gagasan **Aug** secara teoretis lebih kuat: bila tahap kedua mengetahui `S1(x)`, ia
dapat mempelajari sendiri di mana tahap pertama lemah dan seberapa besar koreksi yang
pantas - sebuah **gerbang terpelajar** yang secara ketat lebih umum daripada tabel
`w(s)`. Desainnya faktorial 2x2 sehingga sumbangan masing-masing terpisah.

## Penjinakan overfitting seleksi

Memilih satu `w` per segmen memperbesar ruang seleksi dari 1 menjadi maksimal 14
parameter. Uji coba awal menunjukkan gejalanya nyata: RMSE validation turun, RMSE test
justru naik sampai 3%. Ini persoalan yang sama yang sudah terlihat pada exp05b (V3
dengan `S1 = linear` memburuk 7,5% semata-mata karena `w` ikut dipilih).

Karena itu skema segmentasi dan hyperparameter **tidak** dipilih dari RMSE validation
langsung, melainkan dari **RMSE validasi-silang 5 lipatan kronologis di dalam
validation**. Skema yang hanya mencocokkan derau tidak akan terpilih, dan skema
`global` (setara gerbang skalar) menang secara wajar bila segmentasi tidak membantu.
Peta `w` akhir baru dipasang pada seluruh validation setelah skemanya terpilih.

Test tetap disentuh tepat satu kali.

## Efisiensi

Pembanding (naif, XGBoost polos, kerangka lama, `S1` sendirian, AR-LRX skalar) **tidak
dilatih ulang** - prediksinya dibaca dari `exp05b_..._predictions.npz`. Inilah gunanya
menyimpan prediksi pada exp05b. Yang dilatih hanya 18 model baru.

## Prasyarat

`src/experiments/arlrx.py` versi terbaru (memuat `run_arlrx_segmented`), lalu
**Restart Kernel**. Penambahannya murni fungsi baru; `run_arlrx` tidak disentuh
sehingga exp05a/exp05b tetap tereproduksi persis.

In [1]:
import sys, os, json, warnings
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.experiments import protocol as P
from src.experiments import arlrx as A

P.set_global_seed()
EXPERIMENT = "exp05d_rossmann_arlrx_dev"
REF = "exp05b_rossmann_arlrx_audit"
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60)
print("Lingkungan:", P.environment_stamp())

import inspect
src = inspect.getsource(A)
assert "run_arlrx_segmented" in src and "augment_stage1" in src, (
    "arlrx.py masih versi lama. Ganti berkasnya, lalu Restart Kernel.")
print("arlrx.py: versi dengan gerbang segmen + augmentasi  OK")
print("Skema segmentasi:", A.SEGMENT_SCHEMES, "| lipatan seleksi:", A.N_GATE_FOLDS)

Lingkungan: {'python': '3.10.11', 'platform': 'Windows-10-10.0.22621-SP0', 'numpy': '2.2.6', 'pandas': '2.3.3', 'seed': '42', 'sklearn': '1.7.2', 'xgboost': '3.2.0', 'statsmodels': '0.14.6'}
arlrx.py: versi dengan gerbang segmen + augmentasi  OK
Skema segmentasi: ('global', 'promo', 'dow', 'dow_promo', 'store_q') | lipatan seleksi: 5


## 1. Data, dan pembanding yang dibaca dari exp05b

In [2]:
frame = P.build_rossmann_frame("../data/raw/rossmann/train.csv",
                               "../data/raw/rossmann/store.csv")
VARIANTS = ["V1_customers_dropped", "V2_customers_lagged", "V3_sales_lagged"]
QUICK_RUN = False
if QUICK_RUN:
    keep = np.sort(frame["Store"].unique())[:80]
    frame = frame[frame["Store"].isin(keep)].reset_index(drop=True)
    XGB_GRID = {"n_estimators": [300], "max_depth": [6], "learning_rate": [0.1],
                "subsample": [0.8], "colsample_bytree": [0.8], "max_bin": [256]}
else:
    XGB_GRID = A.GRID_XGB_ARLRX

datasets = {v: P.build_rossmann_dataset(frame, v, target="log1p") for v in VARIANTS}
print("Bentuk kerangka:", frame.shape, "| toko:", frame["Store"].nunique())

ref = pd.read_csv(f"../results/{REF}.csv")
npz = np.load(f"../results/{REF}_predictions.npz")
print(f"Pembanding exp05b dimuat: {len(ref)} baris, {len(npz.files)} larik prediksi")

# sanity: split identik
for v in VARIANTS:
    n_ref = int(ref[ref.feature_set == v].n_test.iloc[0])
    assert n_ref == len(datasets[v].y_test), f"n_test berbeda pada {v}"
    assert np.allclose(npz[f"{v}|y_test"], datasets[v].y_test, atol=1e-5), \
        f"y_test berbeda pada {v} -- exp05b dan exp05d tidak sebanding"
print("Split dan target test identik dengan exp05b  OK")

Bentuk kerangka: (838760, 23) | toko: 1115
Pembanding exp05b dimuat: 27 baris, 48 larik prediksi
Split dan target test identik dengan exp05b  OK


## 2. Menjalankan 18 model baru

Tiga jenis pengembangan x dua tahap pertama x tiga varian. `S1 = linear` sengaja
tidak disertakan: exp05b sudah menunjukkan ia kalah dari XGBoost polos di ketiga
varian, dan mempertahankannya hanya menambah waktu tanpa menambah informasi.

* **AR-LRX-Seg** - gerbang per segmen, tanpa augmentasi
* **AR-LRX-Aug** - augmentasi, gerbang skalar (`schemes=("global",)`)
* **AR-LRX-Aug-Seg** - keduanya

In [3]:
KINDS = ("structural", "struct_linear")
rows = []
for variant in VARIANTS:
    d = datasets[variant]
    for kind in KINDS:
        rows.append(A.run_arlrx_segmented(f"AR-LRX-Seg [{kind}]", d, kind, XGB_GRID,
                                          inverse_transform=np.expm1))
        rows.append(A.run_arlrx_segmented(f"AR-LRX-Aug [{kind}]", d, kind, XGB_GRID,
                                          schemes=("global",), augment_stage1=True,
                                          inverse_transform=np.expm1))
        rows.append(A.run_arlrx_segmented(f"AR-LRX-Aug-Seg [{kind}]", d, kind, XGB_GRID,
                                          augment_stage1=True,
                                          inverse_transform=np.expm1))
        print(f"  {variant:24s} S1={kind:14s} selesai", flush=True)

results = P.save_results(rows, EXPERIMENT)
new_key = {(r["feature_set"], r["model"]): r for r in rows}
print(f"\n{len(results)} baris ditulis ke ../results/{EXPERIMENT}.csv")

  V1_customers_dropped     S1=structural     selesai
  V1_customers_dropped     S1=struct_linear  selesai
  V2_customers_lagged      S1=structural     selesai
  V2_customers_lagged      S1=struct_linear  selesai
  V3_sales_lagged          S1=structural     selesai
  V3_sales_lagged          S1=struct_linear  selesai

18 baris ditulis ke ../results/exp05d_rossmann_arlrx_dev.csv


In [4]:
# Simpan prediksi model baru juga.
store = {}
for (variant, model), r in new_key.items():
    tag = f"{variant}|{model}"
    for k in ("_test_pred", "_stage1_test_raw", "_test_correction", "_test_w"):
        if k in r:
            store[f"{tag}|{k[1:]}"] = np.asarray(r[k], dtype=np.float32)
path = f"../results/{EXPERIMENT}_predictions.npz"
np.savez_compressed(path, **store)
print(f"{len(store)} larik disimpan ({os.path.getsize(path)/1e6:.1f} MB)")

72 larik disimpan (22.6 MB)


## 3. Tabel utama - semua kandidat, satu peringkat

Baris exp05b dimasukkan apa adanya; tidak ada yang dilatih ulang, sehingga
perbandingannya berada di protokol yang sama persis.

In [5]:
def pred_of(variant, model):
    k = f"{variant}|{model}|test_pred"
    if k in npz.files:
        return np.asarray(npz[k], dtype=float)
    return np.asarray(new_key[(variant, model)]["_test_pred"], dtype=float)

REF_MODELS = ["SeasonalNaive(store x dow x promo median)", "XGBoost",
              "LR-XGB (residual, tanpa gerbang)"] + \
             [f"S1 [{k}]" for k in KINDS] + [f"AR-LRX [{k}]" for k in KINDS]

table = []
for variant in VARIANTS:
    sub_ref = ref[ref.feature_set == variant].set_index("model")
    for mname in REF_MODELS:
        table.append({"varian": variant, "model": mname, "asal": "exp05b",
                      "RMSE": sub_ref.loc[mname, "orig_RMSE"],
                      "MAE": sub_ref.loc[mname, "orig_MAE"],
                      "RMSPE": sub_ref.loc[mname, "orig_RMSPE"],
                      "R2": sub_ref.loc[mname, "orig_R2"]})
    for kind in KINDS:
        for pre in ("AR-LRX-Seg", "AR-LRX-Aug", "AR-LRX-Aug-Seg"):
            r = new_key[(variant, f"{pre} [{kind}]")]
            table.append({"varian": variant, "model": f"{pre} [{kind}]", "asal": "exp05d",
                          "RMSE": r["orig_RMSE"], "MAE": r["orig_MAE"],
                          "RMSPE": r["orig_RMSPE"], "R2": r["orig_R2"]})
table = pd.DataFrame(table)
table.to_csv(f"../results/{EXPERIMENT}_main.csv", index=False)

for variant in VARIANTS:
    t = table[table.varian == variant].sort_values("RMSE").reset_index(drop=True)
    t.index = t.index + 1
    print(f"\n--- {variant} ---")
    display(t[["model", "asal", "RMSE", "MAE", "RMSPE", "R2"]].round(5).to_string())


--- V1_customers_dropped ---


'                                        model    asal        RMSE         MAE    RMSPE       R2\n1                  AR-LRX-Aug [struct_linear]  exp05d  1201.35953   811.78630  0.14657  0.85212\n2                      AR-LRX [struct_linear]  exp05b  1204.75075   815.30488  0.14746  0.85129\n3                         AR-LRX [structural]  exp05b  1206.44106   815.91852  0.14739  0.85087\n4              AR-LRX-Aug-Seg [struct_linear]  exp05d  1207.43671   816.79698  0.14766  0.85062\n5                  AR-LRX-Seg [struct_linear]  exp05d  1208.61267   817.48194  0.14818  0.85033\n6                 AR-LRX-Aug-Seg [structural]  exp05d  1211.21720   818.28282  0.14788  0.84969\n7                     AR-LRX-Aug [structural]  exp05d  1219.85044   826.38186  0.14883  0.84754\n8                     AR-LRX-Seg [structural]  exp05d  1220.76472   825.04956  0.14926  0.84731\n9                          S1 [struct_linear]  exp05b  1233.46158   828.52604  0.15329  0.84412\n10                           


--- V2_customers_lagged ---


'                                        model    asal        RMSE        MAE    RMSPE       R2\n1                     AR-LRX-Aug [structural]  exp05d  1026.72738  698.21130  0.12509  0.89199\n2                  AR-LRX-Aug [struct_linear]  exp05d  1029.20832  700.15285  0.12541  0.89147\n3                 AR-LRX-Aug-Seg [structural]  exp05d  1043.40753  714.54182  0.12821  0.88845\n4              AR-LRX-Aug-Seg [struct_linear]  exp05d  1046.97522  718.34719  0.12888  0.88769\n5                      AR-LRX [struct_linear]  exp05b  1054.97223  720.69046  0.13006  0.88597\n6                         AR-LRX [structural]  exp05b  1062.76365  724.32811  0.13087  0.88428\n7                  AR-LRX-Seg [struct_linear]  exp05d  1073.59006  740.17617  0.13362  0.88191\n8                     AR-LRX-Seg [structural]  exp05d  1082.15865  744.22485  0.13430  0.88001\n9                                     XGBoost  exp05b  1111.43912  761.09961  0.13746  0.87343\n10           LR-XGB (residual, tanpa ge


--- V3_sales_lagged ---


'                                        model    asal        RMSE        MAE    RMSPE       R2\n1                     AR-LRX-Aug [structural]  exp05d   945.42129  645.63860  0.11997  0.90842\n2                  AR-LRX-Aug [struct_linear]  exp05d   956.26717  652.96975  0.12126  0.90631\n3              AR-LRX-Aug-Seg [struct_linear]  exp05d   966.10894  666.23821  0.12362  0.90437\n4                 AR-LRX-Aug-Seg [structural]  exp05d   969.23716  666.30962  0.12399  0.90375\n5                      AR-LRX [struct_linear]  exp05b  1004.69483  690.89282  0.12726  0.89658\n6                         AR-LRX [structural]  exp05b  1011.37548  693.35856  0.12838  0.89520\n7                                     XGBoost  exp05b  1017.83105  695.05806  0.13263  0.89385\n8                  AR-LRX-Seg [struct_linear]  exp05d  1025.27535  710.76896  0.13124  0.89230\n9                     AR-LRX-Seg [structural]  exp05d  1034.95268  714.30143  0.13165  0.89025\n10           LR-XGB (residual, tanpa ge

## 4. Ablasi faktorial 2x2 - augmentasi vs segmentasi

Empat sel per (varian, tahap pertama). Sel `skalar` diambil dari exp05b.

In [6]:
cells = []
for variant in VARIANTS:
    sub_ref = ref[ref.feature_set == variant].set_index("model")
    for kind in KINDS:
        base = sub_ref.loc[f"AR-LRX [{kind}]", "orig_RMSE"]
        seg  = new_key[(variant, f"AR-LRX-Seg [{kind}]")]["orig_RMSE"]
        aug  = new_key[(variant, f"AR-LRX-Aug [{kind}]")]["orig_RMSE"]
        both = new_key[(variant, f"AR-LRX-Aug-Seg [{kind}]")]["orig_RMSE"]
        cells.append({"varian": variant, "S1": kind,
                      "skalar (exp05b)": base, "+Seg": seg, "+Aug": aug, "+Aug+Seg": both,
                      "efek Seg (%)": (seg - base) / base * 100,
                      "efek Aug (%)": (aug - base) / base * 100,
                      "efek keduanya (%)": (both - base) / base * 100})
abl = pd.DataFrame(cells)
abl.to_csv(f"../results/{EXPERIMENT}_ablation_2x2.csv", index=False)
print("Negatif = lebih baik daripada gerbang skalar exp05b.")
display(abl.round(3).to_string(index=False))
print("\nRata-rata efek tiap komponen:")
display(abl[["efek Seg (%)", "efek Aug (%)", "efek keduanya (%)"]].mean().round(3))
print("\nSkema segmentasi yang terpilih (perhatikan berapa kali jatuh ke 'global'):")
sc = pd.DataFrame([{"varian": v, "model": m,
                    "skema": r["segment_scheme"], "n_segmen": r["n_segments"],
                    "w": r["gate_w_map"]}
                   for (v, m), r in new_key.items() if "Seg" in m])
display(sc.sort_values(["varian", "model"]).to_string(index=False))

Negatif = lebih baik daripada gerbang skalar exp05b.


'              varian            S1  skalar (exp05b)     +Seg     +Aug  +Aug+Seg  efek Seg (%)  efek Aug (%)  efek keduanya (%)\nV1_customers_dropped    structural         1206.441 1220.765 1219.850  1211.217         1.187         1.111              0.396\nV1_customers_dropped struct_linear         1204.751 1208.613 1201.360  1207.437         0.321        -0.281              0.223\n V2_customers_lagged    structural         1062.764 1082.159 1026.727  1043.408         1.825        -3.391             -1.821\n V2_customers_lagged struct_linear         1054.972 1073.590 1029.208  1046.975         1.765        -2.442             -0.758\n     V3_sales_lagged    structural         1011.375 1034.953  945.421   969.237         2.331        -6.521             -4.166\n     V3_sales_lagged struct_linear         1004.695 1025.275  956.267   966.109         2.048        -4.820             -3.841'


Rata-rata efek tiap komponen:


efek Seg (%)         1.580
efek Aug (%)        -2.724
efek keduanya (%)   -1.661
dtype: float64


Skema segmentasi yang terpilih (perhatikan berapa kali jatuh ke 'global'):


'              varian                          model     skema  n_segmen                                                                                                                            w\nV1_customers_dropped AR-LRX-Aug-Seg [struct_linear] dow_promo        12 {"2": 1.0, "3": 0.5, "4": 0.1, "5": 0.7, "6": 0.5, "7": 0.6, "8": 0.6, "9": 0.6, "10": 0.7, "11": 1.0, "12": 0.9, "14": 1.0}\nV1_customers_dropped    AR-LRX-Aug-Seg [structural] dow_promo        12 {"2": 1.0, "3": 0.6, "4": 0.9, "5": 0.7, "6": 0.3, "7": 0.5, "8": 0.7, "9": 0.5, "10": 0.9, "11": 1.0, "12": 0.9, "14": 1.0}\nV1_customers_dropped     AR-LRX-Seg [struct_linear] dow_promo        12 {"2": 1.0, "3": 0.5, "4": 0.0, "5": 0.7, "6": 0.4, "7": 0.5, "8": 0.5, "9": 0.5, "10": 0.7, "11": 1.0, "12": 0.8, "14": 0.3}\nV1_customers_dropped        AR-LRX-Seg [structural] dow_promo        12 {"2": 1.0, "3": 0.6, "4": 1.0, "5": 0.8, "6": 0.2, "7": 0.5, "8": 0.6, "9": 0.3, "10": 0.9, "11": 1.0, "12": 1.0, "14": 0.5}\n V2_custo

## 5. Uji Diebold-Mariano dua skala untuk model terbaik

Pembandingnya lima: XGBoost polos, naif per toko, kerangka lama, tahap pertama
sendirian, dan **AR-LRX skalar exp05b** - yang terakhir mengukur apakah pengembangan
ini benar-benar melampaui versi sebelumnya, bukan sekadar setara.

In [7]:
best_per_variant = {}
for variant in VARIANTS:
    t = table[(table.varian == variant) & (table.asal == "exp05d")]
    best_per_variant[variant] = t.loc[t.RMSE.idxmin(), "model"]
print("Model exp05d terbaik per varian:")
for v, m in best_per_variant.items():
    print(f"  {v:24s} -> {m}")

dm_rows = []
for variant in VARIANTS:
    d = datasets[variant]
    y_log = d.y_test; y_org = np.expm1(y_log)
    bm = best_per_variant[variant]
    kind = bm.split("[")[1].rstrip("]")
    p_new = pred_of(variant, bm)
    refs = [("XGBoost polos", "XGBoost"),
            ("Naif per toko", "SeasonalNaive(store x dow x promo median)"),
            ("Kerangka lama", "LR-XGB (residual, tanpa gerbang)"),
            (f"S1 [{kind}] sendirian", f"S1 [{kind}]"),
            (f"AR-LRX skalar [{kind}] (exp05b)", f"AR-LRX [{kind}]")]
    for label, mname in refs:
        p_ref = pred_of(variant, mname)
        tl = P.diebold_mariano(y_log, p_new, p_ref)
        to = P.diebold_mariano(y_org, np.expm1(p_new), np.expm1(p_ref))
        dm_rows.append({"varian": variant, "model": bm, "pembanding": label,
                        "DM (log)": round(tl["DM"], 3), "p (log)": tl["p_value"],
                        "DM (asli)": round(to["DM"], 3), "p (asli)": to["p_value"],
                        "menang log": bool(tl["DM"] < 0 and tl["p_value"] < 0.05),
                        "menang asli": bool(to["DM"] < 0 and to["p_value"] < 0.05),
                        "sepakat": bool((tl["DM"] < 0) == (to["DM"] < 0))})
dm = pd.DataFrame(dm_rows)
dm.to_csv(f"../results/{EXPERIMENT}_dm.csv", index=False)
display(dm.to_string(index=False))
print(f"\nSepakat arah      : {int(dm.sepakat.sum())}/{len(dm)}")
print(f"Menang sig (asli) : {int(dm['menang asli'].sum())}/{len(dm)}")
lost = dm[~dm["menang asli"]]
if len(lost):
    print("\nBELUM menang signifikan pada skala asli:")
    display(lost[["varian", "pembanding", "DM (asli)", "p (asli)"]].to_string(index=False))

Model exp05d terbaik per varian:
  V1_customers_dropped     -> AR-LRX-Aug [struct_linear]
  V2_customers_lagged      -> AR-LRX-Aug [structural]
  V3_sales_lagged          -> AR-LRX-Aug [structural]


'              varian                      model                             pembanding  DM (log)  p (log)  DM (asli)     p (asli)  menang log  menang asli  sepakat\nV1_customers_dropped AR-LRX-Aug [struct_linear]                          XGBoost polos   -82.571      0.0    -65.459 0.000000e+00        True         True     True\nV1_customers_dropped AR-LRX-Aug [struct_linear]                          Naif per toko   -40.949      0.0    -33.014 0.000000e+00        True         True     True\nV1_customers_dropped AR-LRX-Aug [struct_linear]                          Kerangka lama   -82.691      0.0    -65.472 0.000000e+00        True         True     True\nV1_customers_dropped AR-LRX-Aug [struct_linear]           S1 [struct_linear] sendirian   -24.696      0.0    -13.297 0.000000e+00        True         True     True\nV1_customers_dropped AR-LRX-Aug [struct_linear] AR-LRX skalar [struct_linear] (exp05b)   -11.863      0.0     -5.729 1.014169e-08        True         True     True\n V2_custo


Sepakat arah      : 15/15
Menang sig (asli) : 15/15


## 6. Apakah kelemahan segmen exp05b tertutup?

exp05b menunjukkan AR-LRX kalah dari XGBoost polos pada segmen tertentu (V3: Selasa,
Kamis, promo, toko kecil). Sel ini memeriksa apakah pengembangan menutupnya.

In [8]:
seg_rows = []
for variant in VARIANTS:
    d = datasets[variant]
    y = np.expm1(d.y_test)
    x = np.expm1(pred_of(variant, "XGBoost"))
    kind = best_per_variant[variant].split("[")[1].rstrip("]")
    old = np.expm1(pred_of(variant, f"AR-LRX [{kind}]"))
    new = np.expm1(pred_of(variant, best_per_variant[variant]))
    fn = list(d.feature_names); Xt = d.X_test
    promo = Xt[:, fn.index("Promo")] if "Promo" in fn else np.zeros(len(y))
    dow = Xt[:, fn.index("DayOfWeek")] if "DayOfWeek" in fn else np.zeros(len(y))
    def rm(t, p, m):
        return float(np.sqrt(np.mean((t[m] - p[m]) ** 2))) if m.sum() else np.nan
    segs = {"semua": np.ones(len(y), bool), "promo": promo == 1, "tanpa promo": promo == 0}
    for dv in sorted(set(np.unique(dow).tolist()))[:7]:
        segs[f"hari {int(dv)}"] = dow == dv
    if "Store" in fn:
        st = Xt[:, fn.index("Store")]
        mean_by_store = pd.Series(y).groupby(pd.Series(st)).transform("mean").to_numpy()
        q = pd.Categorical(pd.qcut(mean_by_store, 4,
                                   labels=["Q1 kecil", "Q2", "Q3", "Q4 besar"],
                                   duplicates="drop"))
        for lab in q.categories:
            segs[f"toko {lab}"] = np.asarray(q == lab)
    for name, m in segs.items():
        ro, rn, rx = rm(y, old, m), rm(y, new, m), rm(y, x, m)
        seg_rows.append({"varian": variant, "segmen": name, "n": int(m.sum()),
                         "AR-LRX lama": ro, "exp05d": rn, "XGBoost": rx,
                         "lama vs XGB (%)": (ro - rx) / rx * 100,
                         "baru vs XGB (%)": (rn - rx) / rx * 100,
                         "perbaikan (pp)": ((rn - rx) / rx * 100) - ((ro - rx) / rx * 100)})
seg = pd.DataFrame(seg_rows)
seg.to_csv(f"../results/{EXPERIMENT}_segments.csv", index=False)
print("Negatif = lebih baik dari XGBoost. 'perbaikan (pp)' negatif = exp05d menutup celah.")
for v in VARIANTS:
    print(f"\n--- {v} ---")
    display(seg[seg.varian == v].round(3).to_string(index=False))
fixed = seg[(seg["lama vs XGB (%)"] > 0) & (seg["baru vs XGB (%)"] < 0)]
print(f"\nSegmen yang berbalik dari kalah menjadi menang: {len(fixed)}")
if len(fixed):
    display(fixed[["varian", "segmen", "n", "lama vs XGB (%)", "baru vs XGB (%)"]]
            .round(3).to_string(index=False))

Negatif = lebih baik dari XGBoost. 'perbaikan (pp)' negatif = exp05d menutup celah.

--- V1_customers_dropped ---


'              varian        segmen      n  AR-LRX lama   exp05d  XGBoost  lama vs XGB (%)  baru vs XGB (%)  perbaikan (pp)\nV1_customers_dropped         semua 129333     1204.751 1201.360 1651.251          -27.040          -27.246          -0.205\nV1_customers_dropped         promo  58399     1365.172 1362.529 1848.483          -26.146          -26.289          -0.143\nV1_customers_dropped   tanpa promo  70934     1054.517 1050.270 1469.133          -28.222          -28.511          -0.289\nV1_customers_dropped        hari 1  20121     1216.093 1211.560 1784.424          -31.850          -32.104          -0.254\nV1_customers_dropped        hari 2  22284     1255.197 1249.287 1682.141          -25.381          -25.732          -0.351\nV1_customers_dropped        hari 3  22281     1192.564 1190.599 1598.009          -25.372          -25.495          -0.123\nV1_customers_dropped        hari 4  20516     1282.206 1282.396 1643.033          -21.961          -21.949           0.012\nV1_cust


--- V2_customers_lagged ---


'             varian        segmen      n  AR-LRX lama   exp05d  XGBoost  lama vs XGB (%)  baru vs XGB (%)  perbaikan (pp)\nV2_customers_lagged         semua 129333     1062.764 1026.727 1111.439           -4.379           -7.622          -3.242\nV2_customers_lagged         promo  58399     1253.268 1218.341 1285.364           -2.497           -5.214          -2.717\nV2_customers_lagged   tanpa promo  70934      875.341  836.662  944.510           -7.323          -11.418          -4.095\nV2_customers_lagged        hari 1  20121     1168.977 1158.577 1313.907          -11.030          -11.822          -0.792\nV2_customers_lagged        hari 2  22284     1126.205 1089.458 1110.729            1.393           -1.915          -3.308\nV2_customers_lagged        hari 3  22281     1106.867 1071.067 1114.326           -0.669           -3.882          -3.213\nV2_customers_lagged        hari 4  20516     1124.969 1067.567 1070.967            5.042           -0.317          -5.360\nV2_customers_la


--- V3_sales_lagged ---


'         varian        segmen      n  AR-LRX lama   exp05d  XGBoost  lama vs XGB (%)  baru vs XGB (%)  perbaikan (pp)\nV3_sales_lagged         semua 129333     1011.375  945.421 1017.831           -0.634           -7.114          -6.480\nV3_sales_lagged         promo  58399     1182.203 1100.701 1151.855            2.635           -4.441          -7.076\nV3_sales_lagged   tanpa promo  70934      845.207  795.138  892.512           -5.300          -10.910          -5.610\nV3_sales_lagged        hari 1  20121     1157.412 1138.731 1273.462           -9.113          -10.580          -1.467\nV3_sales_lagged        hari 2  22284     1051.522  954.891  931.201           12.921            2.544         -10.377\nV3_sales_lagged        hari 3  22281     1021.670  932.357  974.122            4.881           -4.287          -9.169\nV3_sales_lagged        hari 4  20516     1054.411  939.246  921.341           14.443            1.943         -12.500\nV3_sales_lagged        hari 5  21205      885.1


Segmen yang berbalik dari kalah menjadi menang: 8


'              varian        segmen     n  lama vs XGB (%)  baru vs XGB (%)\nV1_customers_dropped        hari 7   640            0.121           -1.302\n V2_customers_lagged        hari 2 22284            1.393           -1.915\n V2_customers_lagged        hari 4 20516            5.042           -0.317\n V2_customers_lagged       toko Q2 32275            0.524           -3.596\n     V3_sales_lagged         promo 58399            2.635           -4.441\n     V3_sales_lagged        hari 3 22281            4.881           -4.287\n     V3_sales_lagged toko Q1 kecil 32421            5.000           -3.151\n     V3_sales_lagged       toko Q2 32275            4.584           -2.907'

## 7. Determinisme dan ringkasan naskah

In [9]:
P.set_global_seed()
v0 = VARIANTS[0]
again = A.run_arlrx_segmented("AR-LRX-Aug [struct_linear]", datasets[v0], "struct_linear",
                              XGB_GRID, schemes=("global",), augment_stage1=True,
                              inverse_transform=np.expm1)
first = new_key[(v0, "AR-LRX-Aug [struct_linear]")]
print("Prediksi identik bit-per-bit:",
      np.array_equal(again["_test_pred"], first["_test_pred"]))
print(f"RMSE: {again['orig_RMSE']:.8f} vs {first['orig_RMSE']:.8f}")

print("\n" + "=" * 88)
for variant in VARIANTS:
    bm = best_per_variant[variant]
    r = new_key[(variant, bm)]
    sub_ref = ref[ref.feature_set == variant].set_index("model")
    kind = bm.split("[")[1].rstrip("]")
    dsub = dm[dm.varian == variant].set_index("pembanding")
    print(f"\n{variant}  ->  {bm}")
    print(f"  RMSE {r['orig_RMSE']:.2f} | MAE {r['orig_MAE']:.2f} | "
          f"RMSPE {r['orig_RMSPE']:.5f} | R2 {r['orig_R2']:.4f}")
    for lab, mname in [("naif per toko", "SeasonalNaive(store x dow x promo median)"),
                       ("XGBoost polos", "XGBoost"),
                       ("kerangka lama", "LR-XGB (residual, tanpa gerbang)"),
                       (f"AR-LRX skalar exp05b", f"AR-LRX [{kind}]")]:
        base = sub_ref.loc[mname, "orig_RMSE"]
        d_ = (r["orig_RMSE"] - base) / base * 100
        key = {"naif per toko": "Naif per toko", "XGBoost polos": "XGBoost polos",
               "kerangka lama": "Kerangka lama",
               "AR-LRX skalar exp05b": f"AR-LRX skalar [{kind}] (exp05b)"}[lab]
        p_ = dsub.loc[key, "p (asli)"]
        arah = "lebih baik" if d_ < 0 else "LEBIH BURUK"
        print(f"    {arah} {abs(d_):5.2f}% dari {lab:26s} (DM asli, p = {p_:.3g})")
print("\n" + "=" * 88)
print("Berkas yang dihasilkan:")
for f in sorted(os.listdir("../results")):
    if f.startswith(EXPERIMENT):
        print(f"  ../results/{f}")

Prediksi identik bit-per-bit: True
RMSE: 1201.35953098 vs 1201.35953098


V1_customers_dropped  ->  AR-LRX-Aug [struct_linear]
  RMSE 1201.36 | MAE 811.79 | RMSPE 0.14657 | R2 0.8521
    lebih baik  5.75% dari naif per toko              (DM asli, p = 0)
    lebih baik 27.25% dari XGBoost polos              (DM asli, p = 0)
    lebih baik 27.43% dari kerangka lama              (DM asli, p = 0)
    lebih baik  0.28% dari AR-LRX skalar exp05b       (DM asli, p = 1.01e-08)

V2_customers_lagged  ->  AR-LRX-Aug [structural]
  RMSE 1026.73 | MAE 698.21 | RMSPE 0.12509 | R2 0.8920
    lebih baik 19.45% dari naif per toko              (DM asli, p = 0)
    lebih baik  7.62% dari XGBoost polos              (DM asli, p = 0)
    lebih baik 10.74% dari kerangka lama              (DM asli, p = 0)
    lebih baik  3.39% dari AR-LRX skalar exp05b       (DM asli, p = 0)

V3_sales_lagged  ->  AR-LRX-Aug [structural]
  RMSE 945.42 | MAE 645.64 | RMSPE 0.11997 | R2 0.9084
    lebih baik 25.83% dari naif per

## 8. Cara melaporkan

* **Desainnya faktorial, jadi laporkan keempat selnya** - termasuk bila `+Seg`
  memburuk. Justru kontras "gerbang tabel gagal, gerbang terpelajar berhasil" yang
  membuat temuannya bermakna: yang menentukan bukan banyaknya parameter gerbang,
  melainkan apakah gerbang itu dapat bergantung pada fitur secara kontinu.
* **Sebutkan penjinakan overfitting seleksi.** Bahwa skema segmentasi dipilih lewat
  validasi-silang di dalam validation, dan bahwa tanpa itu RMSE test naik sampai 3%,
  adalah bukti kedisiplinan metodologis yang jarang ditunjukkan naskah sejenis.
* **Sebutkan keterbatasan augmentasi.** `S1` pada blok fit adalah nilai in-sample;
  untuk penaksir struktural nilai itu sedikit optimistis karena rata-rata kelompok
  memuat titiknya sendiri. Konvensi ini sama dengan yang dipakai membentuk residual
  pada kerangka aslinya, tetapi tetap harus dinyatakan.
* **Jangan buang exp05b.** Ia yang membuktikan reproduksi persis, kesepakatan dua
  skala, dan ablasi gerbang skalar. exp05d berdiri di atasnya, bukan menggantikannya.